In [1]:
import requests 
from dotenv import load_dotenv
import os 
load_dotenv()

#getting API key from env
api_key = os.getenv("COMPANIES_HOUSE_API_KEY")
if not api_key:
    print("COMPANIES_HOUSE_API_KEY is not set")

URL = "https://document-api.company-information.service.gov.uk/document/9N7sJvwmt0UtuWyTEEZ4q3BRBLLktpAuRGa4ZCxRSxI"
#testing requests 

document_metadata = requests.get(URL, auth=(api_key,''))

print(document_metadata.status_code)



200


In [2]:
document_metadata.json()


{'company_number': '05150526',
 'barcode': 'XEXYI9JT',
 'significant_date': '2025-09-30T00:00:00Z',
 'significant_date_type': 'made-up-date',
 'category': 'accounts',
 'pages': 4,
 'filename': '05150526_aa_2026-03-16',
 'created_at': '2026-03-16T12:27:43.122466296Z',
 'etag': '',
 'links': {'self': 'https://document-api.company-information.service.gov.uk/document/9N7sJvwmt0UtuWyTEEZ4q3BRBLLktpAuRGa4ZCxRSxI',
  'document': 'https://document-api.company-information.service.gov.uk/document/9N7sJvwmt0UtuWyTEEZ4q3BRBLLktpAuRGa4ZCxRSxI/content'},
 'resources': {'application/pdf': {'content_length': 30752},
  'application/xhtml+xml': {'content_length': 34960}}}

In [3]:
link_document = document_metadata.json()['links']['document']
headers = {
    "Accept": "application/xhtml+xml"
}

document_response = requests.get(link_document, 
                                 auth=(api_key,''),
                                 headers = headers
)


print(document_response.status_code)

document_response.headers['Content-Type']


200


'application/xhtml+xml'

In [4]:
from bs4 import BeautifulSoup
document_soup = BeautifulSoup(document_response.text, 'html.parser')
document_tag = document_soup.find("ix:nonfraction")

#find the current assets
current_asset_tag = document_soup.find(
    "ix:nonfraction",
    attrs={"name": "frs-core:CurrentAssets"}
           )
current_asset_tag.text                    


'1,321'

In [5]:
document_tag["contextref"]

'CURRENT_FY_END'

In [6]:
soup_context = document_soup.find("xbrli:context",id='CURRENT_FY_END')
soup_context

<xbrli:context id="CURRENT_FY_END">
<xbrli:entity>
<xbrli:identifier scheme="http://www.companieshouse.gov.uk/">05150526</xbrli:identifier>
</xbrli:entity>
<xbrli:period>
<xbrli:instant>2025-09-30</xbrli:instant>
</xbrli:period>
</xbrli:context>

In [83]:
for tag in document_soup.find_all("ix:nonfraction"):
    if "1,321" in tag.text:
        print(tag)

<ix:nonfraction contextref="CURRENT_FY_END" decimals="0" format="ixt:numcommadot" name="frs-core:CurrentAssets" unitref="GBP">1,321</ix:nonfraction>


In [7]:
 #get filing history of scci

company_number = "05150526"

filing_history_url = f"https://api.company-information.service.gov.uk/company/{company_number}/filing-history"

filing_history = requests.get(filing_history_url, auth=(api_key,''))
filing_history_document = filing_history.json()
filing_history_document['items'][9]['description']

'confirmation-statement-with-no-updates'

In [36]:
url = link_document_previous_account_text

old_document_metadata = requests.get(url, auth=(api_key,''))

from bs4 import BeautifulSoup

old_document_soup = BeautifulSoup(
    old_document_metadata.content,
    "html.parser",
)




Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


In [23]:

def get_full_account(filing_history_document):
    filing = []
    for filings in filing_history_document['items']:
        
    
        if filings['description'] == 'accounts-with-accounts-type-total-exemption-full':
            filing = filings
            return filing

    
scci_filling_full_account = get_full_account(filing_history_document)
scci_full_account_link = scci_filling_full_account['links']['document_metadata']


scci_full_accounts_response = requests.get(scci_full_account_link, auth=(api_key,''))
scci_full_accounts_text = scci_full_accounts_response.json()

link_document_previous_account_text = scci_full_accounts_text['links']['document']



AttributeError: 'str' object has no attribute 'json'

In [37]:
content_url = scci_full_accounts_response.json()["links"]["document"]

xhtml_response = requests.get(
    content_url,
    auth=(api_key, ""),
    headers={"Accept": "application/xhtml+xml"},
)

print("Status:", xhtml_response.status_code)
print("Content type:", xhtml_response.headers.get("Content-Type"))
print("First bytes:", xhtml_response.content[:20])

old_document_soup = BeautifulSoup(
    xhtml_response.content,
    "html.parser",
)

financial_tags = old_document_soup.find_all("ix:nonfraction")

print("Financial tags found:", len(financial_tags))

Status: 200
Content type: application/xhtml+xml
First bytes: b'<?xml version="1.0" '
Financial tags found: 36


In [39]:
for tag in financial_tags:
    print(
        "Concept:", tag.get("name"),
        "| Value:", tag.get_text(strip=True),
        "| Context:", tag.get("contextref"),
        "| Unit:", tag.get("unitref"),
    )

Concept: ns6:AverageNumberEmployeesDuringPeriod | Value: 0 | Context: FY_31_03_2020 | Unit: pure
Concept: ns6:AverageNumberEmployeesDuringPeriod | Value: 0 | Context: FY_31_03_2019 | Unit: pure
Concept: ns6:PropertyPlantEquipment | Value: 20 | Context: cfwd_31_03_2020 | Unit: GBP
Concept: ns6:PropertyPlantEquipment | Value: 30 | Context: cfwd_31_03_2019 | Unit: GBP
Concept: ns6:Debtors | Value: 93 | Context: cfwd_31_03_2020 | Unit: GBP
Concept: ns6:Debtors | Value: 169 | Context: cfwd_31_03_2019 | Unit: GBP
Concept: ns6:CashBankOnHand | Value: 1,150 | Context: cfwd_31_03_2020 | Unit: GBP
Concept: ns6:CashBankOnHand | Value: 2,083 | Context: cfwd_31_03_2019 | Unit: GBP
Concept: ns6:CurrentAssets | Value: 1,243 | Context: cfwd_31_03_2020 | Unit: GBP
Concept: ns6:CurrentAssets | Value: 2,252 | Context: cfwd_31_03_2019 | Unit: GBP
Concept: ns6:Creditors | Value: 7,923 | Context: CreditorsHypercube_cfwd_31_03_2020_Set1 | Unit: GBP
Concept: ns6:Creditors | Value: 6,166 | Context: CreditorsHy

In [40]:
for tag in financial_tags:
    value = tag.get_text(strip=True)
    sign = tag.get("sign")

    if sign == "-" and value != "-":
        value = f"-{value}"

    print(
        "Concept:", tag.get("name"),
        "| Value:", value,
        "| Context:", tag.get("contextref"),
        "| Unit:", tag.get("unitref"),
    )

Concept: ns6:AverageNumberEmployeesDuringPeriod | Value: 0 | Context: FY_31_03_2020 | Unit: pure
Concept: ns6:AverageNumberEmployeesDuringPeriod | Value: 0 | Context: FY_31_03_2019 | Unit: pure
Concept: ns6:PropertyPlantEquipment | Value: 20 | Context: cfwd_31_03_2020 | Unit: GBP
Concept: ns6:PropertyPlantEquipment | Value: 30 | Context: cfwd_31_03_2019 | Unit: GBP
Concept: ns6:Debtors | Value: 93 | Context: cfwd_31_03_2020 | Unit: GBP
Concept: ns6:Debtors | Value: 169 | Context: cfwd_31_03_2019 | Unit: GBP
Concept: ns6:CashBankOnHand | Value: 1,150 | Context: cfwd_31_03_2020 | Unit: GBP
Concept: ns6:CashBankOnHand | Value: 2,083 | Context: cfwd_31_03_2019 | Unit: GBP
Concept: ns6:CurrentAssets | Value: 1,243 | Context: cfwd_31_03_2020 | Unit: GBP
Concept: ns6:CurrentAssets | Value: 2,252 | Context: cfwd_31_03_2019 | Unit: GBP
Concept: ns6:Creditors | Value: 7,923 | Context: CreditorsHypercube_cfwd_31_03_2020_Set1 | Unit: GBP
Concept: ns6:Creditors | Value: 6,166 | Context: CreditorsHy

In [3]:
import json
import sqlite3
from pathlib import Path
import pandas as pd 



BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"
DB_PATH = BASE_DIR / "data" / "database" / "cf.db"



with sqlite3.connect(DB_PATH) as conn:
    cursor = conn.cursor()

    cursor.execute("""

        CREATE TABLE IF NOT EXISTS financial_facts(
            fact_id INTEGER PRIMARY KEY AUTOINCREMENT,
            company_number TEXT NOT NULL,
            document_id TEXT NOT NULL,
            concept TEXT NOT NULL, 
            concept_local_name TEXT NOT NULL, 
            value_raw TEXT,
            value_numeric NUMERIC, 
            unit TEXT,
            context_ref TEXT NOT NULL,
            source_url TEXT,
            UNIQUE (
                company_number,
                document_id,
                concept,
                context_ref
            )
        )
    """)

    conn.commit()

#dict created for companies name

companies_name = {
    'jlas': 'jlas-development-capital',
    'cadence': 'cadence-equity-partners',
    'scci': 'scci',
}


#retrieving company info
json_file_name = f"company_data_{companies_name['cadence']}-limited.json"

BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"
DB_PATH = BASE_DIR / "data" / "database" / "cf.db"

with open(DATA_DIR / json_file_name) as f:
    data = json.load(f)
company_name = data["company_profile"]["company_name"]
profile = data['company_profile']
company_number = profile['company_number']
document_id = content_url.split("/document/")[1].split("/")[0]

#filling the table 

with sqlite3.connect(DB_PATH) as conn:
    cursor = conn.cursor()

    for tag in financial_tags:
        concept = tag.get("name")
        concept_local_name = concept.split(":")[-1]

        value_raw = tag.get_text(strip=True)
        sign = tag.get("sign")

        if value_raw in ("", "-"):
            value_numeric = None
        else:
            value_numeric = value_raw.replace(",", "")

            if sign == "-":
                value_numeric = f"-{value_numeric}"

        unit = tag.get('unitref')
        context_ref = tag.get('contextref')
        source_url = content_url

        cursor.execute("""
    
            INSERT OR IGNORE INTO financial_facts (
                company_number,
                document_id,
                concept, 
                concept_local_name, 
                value_raw,
                value_numeric, 
                unit,
                context_ref,
                source_url
            )
            VALUES (?,?,?,?,?,?,?,?,?)
        """, (
            company_number,
            document_id,
            concept,
            concept_local_name,
            value_raw,
            value_numeric,
            unit,
            context_ref,
            source_url,
        ))
    
conn.commit()  
        



conn.close()
            

NameError: name 'content_url' is not defined

In [81]:
with sqlite3.connect(DB_PATH) as conn:
     financial_facts_df = pd.read_sql_query("""
        SELECT
            fact_id,
            concept_local_name,
            value_numeric,
            unit,
            context_ref
        FROM financial_facts
        WHERE company_number = ?
        ORDER BY fact_id
    """, conn, params=(company_number,))

financial_facts_df





,fact_id,concept_local_name,value_numeric,unit,context_ref
0,1,Creditors,6166.0,GBP,CreditorsHypercube_cfwd_31_03_2019_Set1
1,2,AverageNumberEmployeesDuringPeriod,0.0,pure,FY_31_03_2020
2,3,AverageNumberEmployeesDuringPeriod,0.0,pure,FY_31_03_2019
3,4,PropertyPlantEquipment,20.0,GBP,cfwd_31_03_2020
4,5,PropertyPlantEquipment,30.0,GBP,cfwd_31_03_2019
5,6,Debtors,93.0,GBP,cfwd_31_03_2020
6,7,Debtors,169.0,GBP,cfwd_31_03_2019
7,8,CashBankOnHand,1150.0,GBP,cfwd_31_03_2020
8,9,CashBankOnHand,2083.0,GBP,cfwd_31_03_2019
9,10,CurrentAssets,1243.0,GBP,cfwd_31_03_2020


In [82]:
with sqlite3.connect(DB_PATH) as conn:
    financial_facts_df = pd.read_sql_query("""
        SELECT
            fact_id,
            concept_local_name
            value_numeric,
            unit,
            context_ref
        FROM financial_facts
        ORDER BY fact_id 
        """,
        conn,
        params=(company_number,))
    

financial_facts_df



DatabaseError: Execution failed on sql '
        SELECT
            fact_id,
            concept_local_name
            value_numeric,
            unit,
            context_ref
        FROM financial_facts
        ORDER BY fact_id 
        ': Incorrect number of bindings supplied. The current statement uses 0, and there are 1 supplied.